# 多元线性回归完整教程：从简单回归到多变量建模

## 📚 教学目标
1. 理解多元回归方程的结构和各系数的含义
2. 掌握虚拟变量（哑变量）的使用方法
3. 理解 R² vs 调整 R² 的区别及其在模型评价中的作用
4. 识别多重共线性问题及其诊断方法
5. 用 statsmodels 进行完整的多元回归分析

**参考书**：《基础统计学(第14版)》（Triola）第 10-4 节
**教学策略**：先用简单线性回归回顾基础，再逐步引入多个自变量和虚拟变量

---

## 1. 场景设定：预测房价

### 🎯 问题

我们想预测房屋价格，已知以下变量：
- 房屋面积（平方英尺）
- 卧室数量
- 是否有游泳池（分类变量）
- 房龄（年）

### 📖 书中的定义

> 多元回归方程表示一个因变量与两个或两个以上自变量之间的线性关系。
> 多元回归方程的一般形式为：
> $$\hat{y} = b_0 + b_1 x_1 + b_2 x_2 + \cdots + b_k x_k$$

### 📐 多元回归方程

$$\hat{y} = b_0 + b_1 x_1 + b_2 x_2 + b_3 x_3$$

其中：
- $\hat{y}$ = 因变量的预测值（房屋价格）
- $b_0$ = 截距（所有自变量为 0 时的预测值）
- $b_1, b_2, b_3$ = 偏回归系数（其他变量不变时，该变量每增加 1 单位，y 的变化量）
- $x_1, x_2, x_3$ = 自变量

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 数据生成与探索

### 📐 数据生成过程 (DGP)

我们模拟一个房价数据集：
- 面积 (sqft): 正态分布, μ=1500, σ=500
- 卧室数 (bedrooms): 均匀分布, 2-5
- 游泳池 (pool): 伯努利分布, p=0.3
- 房龄 (age): 均匀分布, 1-30 年
- 真实关系: price = 50000 + 100×sqft + 15000×bedrooms + 25000×pool - 1000×age + ε

In [ ]:
# ========== 步骤 1: 生成模拟数据 ==========

n = 200

# --- 数据生成过程 (DGP) ---
# 自变量1: 房屋面积 (平方英尺)
sqft = np.random.normal(1500, 500, n)
sqft = np.clip(sqft, 500, 3500)

# 自变量2: 卧室数量
bedrooms = np.random.randint(2, 6, n)

# 自变量3: 是否有游泳池 (0/1 虚拟变量)
pool = np.random.binomial(1, 0.3, n)

# 自变量4: 房龄
age = np.random.randint(1, 31, n)

# 因变量: 房价 (真实关系 + 随机误差)
noise = np.random.normal(0, 30000, n)
price = 50000 + 100 * sqft + 15000 * bedrooms + 25000 * pool - 1000 * age + noise

# 构建 DataFrame
df = pd.DataFrame({
    'price': price,
    'sqft': sqft,
    'bedrooms': bedrooms,
    'pool': pool,
    'age': age
})

print("📊 步骤 1: 数据生成")
print(f"  样本量 n = {n}")
print(f"  自变量: sqft, bedrooms, pool (虚拟变量), age")
print(f"  因变量: price")
print(f"\n📊 数据前 5 行:")
print(df.head().to_string(index=False))
print(f"\n📊 基本统计量:")
print(df.describe().round(2).to_string())

---

## 3. 简单线性回归回顾

### 📐 简单线性回归方程

$$\hat{y} = b_0 + b_1 x_1$$

先用面积 (sqft) 单独预测房价，作为多元回归的对比基准。

In [ ]:
# ========== 步骤 2: 简单线性回归 (仅 sqft) ==========

print("=" * 60)
print("📋 简单线性回归: price ~ sqft")
print("=" * 60)

# 手算回归系数
x = df['sqft'].values
y = df['price'].values
n_obs = len(y)

x_mean = np.mean(x)
y_mean = np.mean(y)

# b1 = Σ(xi - x̄)(yi - ȳ) / Σ(xi - x̄)²
b1 = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean)**2)
b0 = y_mean - b1 * x_mean

print(f"\n📊 步骤 2: 手算回归系数")
print(f"  x̄ (sqft 均值) = {x_mean:.2f}")
print(f"  ȳ (price 均值) = {y_mean:.2f}")
print(f"  b₁ (斜率) = {b1:.4f}")
print(f"  b₀ (截距) = {b0:.4f}")
print(f"\n  方程: ŷ = {b0:.2f} + {b1:.2f} × sqft")

# 预测值和残差
y_hat = b0 + b1 * x
residuals = y - y_hat
ss_res = np.sum(residuals**2)
ss_tot = np.sum((y - y_mean)**2)
r_squared = 1 - ss_res / ss_tot

print(f"\n📊 模型拟合指标")
print(f"  残差平方和 SS_res = {ss_res:.2f}")
print(f"  总平方和 SS_tot = {ss_tot:.2f}")
print(f"  R² = {r_squared:.4f}")
print(f"  即 sqft 解释了房价变异的 {r_squared*100:.1f}%")

In [ ]:
# ========== 可视化: 简单线性回归 ==========

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 散点图 + 回归线
ax1 = axes[0]
ax1.scatter(df['sqft'], df['price'], alpha=0.4, color='steelblue', s=20, label='Data Points')
x_line = np.linspace(df['sqft'].min(), df['sqft'].max(), 100)
ax1.plot(x_line, b0 + b1 * x_line, 'r-', linewidth=2, label=f'Regression Line')
ax1.set_xlabel('Square Footage', fontsize=12)
ax1.set_ylabel('Price ($)', fontsize=12)
ax1.set_title('Simple Linear Regression: Price vs Sqft', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# 残差图
ax2 = axes[1]
ax2.scatter(y_hat, residuals, alpha=0.4, color='steelblue', s=20)
ax2.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
ax2.set_xlabel('Predicted Values', fontsize=12)
ax2.set_ylabel('Residuals', fontsize=12)
ax2.set_title('Residual Plot', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  左图：散点图显示 sqft 与 price 的正相关关系")
print(f"  R² = {r_squared:.4f}，仅用面积解释了 {r_squared*100:.1f}% 的房价变异")
print("  右图：残差图用于检查模型假设（残差应随机分布在 0 附近）")

---

## 4. 多元线性回归

### 📐 多元回归方程

$$\hat{y} = b_0 + b_1 x_1 + b_2 x_2 + b_3 x_3 + b_4 x_4$$

其中：
- $x_1$ = sqft（面积）
- $x_2$ = bedrooms（卧室数）
- $x_3$ = pool（游泳池，虚拟变量）
- $x_4$ = age（房龄）

### 💡 虚拟变量

对于分类变量（如有/无游泳池），使用 0/1 编码：
- pool = 1: 有游泳池
- pool = 0: 无游泳池

系数 $b_3$ 的含义：在其他变量相同的情况下，有游泳池的房屋比没有游泳池的房屋平均贵 $b_3$ 美元。

In [ ]:
# ========== 步骤 3: 多元线性回归 ==========

print("=" * 60)
print("📋 多元线性回归: price ~ sqft + bedrooms + pool + age")
print("=" * 60)

# 使用 statsmodels 进行多元回归
X = df[['sqft', 'bedrooms', 'pool', 'age']]
X_with_const = sm.add_constant(X)  # 添加截距项
y = df['price']

model = sm.OLS(y, X_with_const).fit()

print(f"\n📊 回归系数:")
print(f"{'变量':<12} {'系数':<15} {'标准误':<12} {'t 值':<10} {'p 值':<12}")
print("-" * 61)
for var in model.params.index:
    print(f"{var:<12} {model.params[var]:<15.4f} {model.bse[var]:<12.4f} {model.tvalues[var]:<10.4f} {model.pvalues[var]:<12.6f}")

print(f"\n📊 模型拟合指标:")
print(f"  R² = {model.rsquared:.4f}")
print(f"  调整 R² = {model.rsquared_adj:.4f}")
print(f"  F 统计量 = {model.fvalue:.4f}")
print(f"  F 检验 p 值 = {model.f_pvalue:.6f}")
print(f"  殂数标准误 = {np.sqrt(model.mse_resid):.4f}")

print(f"\n📊 回归方程:")
print(f"  ŷ = {model.params['const']:.2f} + {model.params['sqft']:.2f}×sqft + {model.params['bedrooms']:.2f}×bedrooms")
print(f"      + {model.params['pool']:.2f}×pool + {model.params['age']:.2f}×age")

In [ ]:
# ========== 步骤 4: scipy 验证 ==========

print("=" * 60)
print("🔬 验证: 对比简单回归 vs 多元回归")
print("=" * 60)

# 简单回归 R² (手算)
print(f"\n📊 R² 对比:")
print(f"  简单回归 (仅 sqft): R² = {r_squared:.4f}")
print(f"  多元回归 (4 个变量): R² = {model.rsquared:.4f}")
print(f"  调整 R² = {model.rsquared_adj:.4f}")

print(f"\n💡 解读:")
print(f"  加入 bedrooms, pool, age 后，R² 从 {r_squared:.4f} 提高到 {model.rsquared:.4f}")
print(f"  调整 R² ({model.rsquared_adj:.4f}) 考虑了变量数量的惩罚")
print(f"  如果 R² 远大于调整 R²，可能存在过拟合风险")

---

## 5. R² vs 调整 R²

### 📐 R² 的公式

$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}}$$

### 📐 调整 R² 的公式

$$R^2_{adj} = 1 - \frac{SS_{res} / (n - k - 1)}{SS_{tot} / (n - 1)} = 1 - (1 - R^2) \frac{n - 1}{n - k - 1}$$

其中：
- n = 样本量
- k = 自变量个数

### 💡 关键区别

| 指标 | R² | 调整 R² |
|------|------|----------|
| 增加变量 | 总是增加或不变 | 可能增加也可能减少 |
| 惩罚 | 无 | 惩罚过多变量 |
| 用途 | 衡量拟合优度 | 比较不同变量数的模型 |

In [ ]:
# ========== 步骤 5: R² vs 调整 R² 演示 ==========

print("=" * 60)
print("📋 R² vs 调整 R²: 逐步增加变量")
print("=" * 60)

# 逐步增加变量，观察 R² 和调整 R² 的变化
variable_sets = [
    ['sqft'],
    ['sqft', 'bedrooms'],
    ['sqft', 'bedrooms', 'pool'],
    ['sqft', 'bedrooms', 'pool', 'age'],
]

print(f"\n{'变量组合':<35} {'R²':<10} {'调整 R²':<10}")
print("-" * 55)

for vars_set in variable_sets:
    X_sub = sm.add_constant(df[vars_set])
    model_sub = sm.OLS(y, X_sub).fit()
    var_str = ', '.join(vars_set)
    print(f"  {var_str:<33} {model_sub.rsquared:<10.4f} {model_sub.rsquared_adj:<10.4f}")

print(f"\n💡 关键发现:")
print(f"  R² 随变量增加而单调递增（不会减少）")
print(f"  调整 R² 只在变量有贡献时才会增加")
print(f"  如果加入某变量后调整 R² 下降，说明该变量可能无用")

---

## 6. 多重共线性诊断

### 📐 方差膨胀因子 (VIF)

$$VIF_j = \frac{1}{1 - R_j^2}$$

其中 $R_j^2$ 是用第 j 个自变量对其余自变量回归得到的 R²。

### 💡 VIF 解读

| VIF 值 | 共线性程度 |
|--------|------------|
| 1 | 无共线性 |
| 1-5 | 轻度共线性 |
| 5-10 | 中度共线性 |
| > 10 | 严重共线性 |

In [ ]:
# ========== 步骤 6: 多重共线性诊断 ==========

print("=" * 60)
print("📋 多重共线性诊断: VIF 计算")
print("=" * 60)

# 计算 VIF
X_vif = df[['sqft', 'bedrooms', 'pool', 'age']]
X_vif_with_const = sm.add_constant(X_vif)

vif_data = pd.DataFrame()
vif_data['Variable'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif_with_const.values, i+1) 
                    for i in range(len(X_vif.columns))]

print(f"\n📊 方差膨胀因子 (VIF):")
print(f"{'变量':<12} {'VIF':<10} {'共线性程度':<15}")
print("-" * 37)
for _, row in vif_data.iterrows():
    vif_val = row['VIF']
    if vif_val < 5:
        level = '轻度'
    elif vif_val < 10:
        level = '中度'
    else:
        level = '严重'
    print(f"  {row['Variable']:<12} {vif_val:<10.4f} {level}")

print(f"\n💡 解读:")
print(f"  VIF = 1 表示该变量与其他自变量完全不相关")
print(f"  VIF > 10 表示存在严重共线性，应考虑删除变量")
print(f"  本例中所有 VIF 都接近 1，说明变量间几乎无共线性")

---

## 7. 残差分析与模型诊断

### 📐 残差的定义

$$e_i = y_i - \hat{y}_i$$

### 📐 标准化残差

$$z_i = \frac{e_i}{s_e}$$

其中 $s_e$ 是估计标准误。

In [ ]:
# ========== 步骤 7: 残差分析 ==========

print("=" * 60)
print("📋 残差分析与模型诊断")
print("=" * 60)

# 获取残差和标准化残差
residuals_multi = model.resid
std_residuals = model.get_influence().resid_studentized_internal
fitted_values = model.fittedvalues

# 强影响点检测 (Cook's Distance)
influence = model.get_influence()
cooks_d = influence.cooks_distance[0]

print(f"\n📊 残差统计:")
print(f"  殂差均值 = {np.mean(residuals_multi):.4f} (应接近 0)")
print(f"  殂差标准差 = {np.std(residuals_multi):.4f}")
print(f"  最大标准化残差 = {np.max(np.abs(std_residuals)):.4f}")

# 强影响点
influential = np.where(cooks_d > 4/n)[0]
print(f"\n📊 强影响点 (Cook's D > 4/n = {4/n:.4f}):")
print(f"  发现 {len(influential)} 个强影响点")
if len(influential) > 0:
    print(f"  索引: {influential[:10].tolist()}")

In [ ]:
# ========== 残差诊断图 ==========

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 残差 vs 拟合值
ax1 = axes[0, 0]
ax1.scatter(fitted_values, residuals_multi, alpha=0.4, color='steelblue', s=20)
ax1.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
ax1.set_xlabel('Fitted Values', fontsize=12)
ax1.set_ylabel('Residuals', fontsize=12)
ax1.set_title('Residuals vs Fitted', fontsize=14, fontweight='bold')
ax1.grid(alpha=0.3)

# 2. QQ 图
ax2 = axes[0, 1]
stats.probplot(residuals_multi, dist="norm", plot=ax2)
ax2.set_title('Normal Q-Q Plot', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3)

# 3. 标准化残差直方图
ax3 = axes[1, 0]
ax3.hist(std_residuals, bins=20, density=True, alpha=0.6, color='steelblue', edgecolor='white')
x_norm = np.linspace(-4, 4, 100)
ax3.plot(x_norm, stats.norm.pdf(x_norm), 'r-', linewidth=2, label='Standard Normal')
ax3.set_xlabel('Standardized Residuals', fontsize=12)
ax3.set_ylabel('Density', fontsize=12)
ax3.set_title('Distribution of Standardized Residuals', fontsize=14, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(alpha=0.3)

# 4. Cook's Distance
ax4 = axes[1, 1]
ax4.stem(range(len(cooks_d)), cooks_d, markerfmt=',', basefmt='k-', linefmt='steelblue')
ax4.axhline(y=4/n, color='red', linestyle='--', linewidth=1.5, label=f'Threshold (4/n = {4/n:.4f})')
ax4.set_xlabel('Observation Index', fontsize=12)
ax4.set_ylabel("Cook's Distance", fontsize=12)
ax4.set_title("Cook's Distance", fontsize=14, fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  1. 残差 vs 拟合值：检查线性和同方差性（应随机分布在 0 附近）")
print("  2. QQ 图：检查残差正态性（点应落在对角线上）")
print("  3. 标准化残差直方图：直观检查残差分布形态")
print("  4. Cook's Distance：识别对模型有过度影响的观测点")

---

## 8. 模型预测

### 💡 用拟合好的模型进行预测

In [ ]:
# ========== 步骤 8: 模型预测 ==========

print("=" * 60)
print("📋 模型预测示例")
print("=" * 60)

# 新数据
new_houses = pd.DataFrame({
    'const': 1,
    'sqft': [1200, 2000, 2500],
    'bedrooms': [3, 4, 5],
    'pool': [0, 1, 1],
    'age': [10, 5, 2]
})

# 预测
predictions = model.predict(new_houses)
pred_ci = model.get_prediction(new_houses).summary_frame(alpha=0.05)

print(f"\n📊 新房屋预测:")
print(f"{'房屋':<6} {'面积':<8} {'卧室':<6} {'泳池':<6} {'房龄':<6} {'预测价格':<15} {'95% CI 下限':<15} {'95% CI 上限':<15}")
print("-" * 77)
for i in range(len(new_houses)):
    print(f"  {i+1:<6} {new_houses['sqft'].iloc[i]:<8.0f} {new_houses['bedrooms'].iloc[i]:<6} {new_houses['pool'].iloc[i]:<6} {new_houses['age'].iloc[i]:<6} ${predictions.iloc[i]:<14,.0f} ${pred_ci['mean_ci_lower'].iloc[i]:<14,.0f} ${pred_ci['mean_ci_upper'].iloc[i]:<14,.0f}")

print(f"\n💡 解读:")
print(f"  预测价格是点估计")
print(f"  95% CI 是均值的置信区间（平均房价的范围）")
print(f"  预测区间会更宽（单个房屋价格的范围）")

---

## 📌 核心概念回顾

### 📌 多元回归方程 (Multiple Regression Equation)
- **定义**: 一个因变量与多个自变量之间的线性关系
- **公式**: $\hat{y} = b_0 + b_1 x_1 + b_2 x_2 + \cdots + b_k x_k$
- **含义**: 每个系数 $b_j$ 表示在其他变量不变时，$x_j$ 每增加 1 单位，y 的平均变化量

### 📌 虚拟变量 (Dummy Variable)
- **定义**: 用 0/1 编码表示分类变量
- **公式**: pool = 1 (有) 或 0 (无)
- **含义**: 系数表示该类别相对于参考类别的平均差异
- **注意**: k 个类别只需 k-1 个虚拟变量

### 📌 R² vs 调整 R²
- **R²**: 衡量模型解释的变异比例，随变量增加而单调递增
- **调整 R²**: 对变量数量进行惩罚，用于比较不同变量数的模型
- **判断标准**: 调整 R² 越大越好

### 📌 多重共线性 (Multicollinearity)
- **定义**: 自变量之间存在高度相关性
- **诊断**: VIF > 10 表示严重共线性
- **影响**: 系数估计不稳定，标准误增大
- **处理**: 删除高度相关的变量之一，或使用主成分回归

### 🔗 完整流程
```
数据准备 → 简单回归 → 多元回归 → 共线性诊断 → 残差分析 → 预测
    ↓          ↓          ↓          ↓           ↓         ↓
  编码变量    基准R²     全模型     VIF<10     QQ图/残差图  新数据
```

---

## ❌ 常见误区

### ❌ 误区 1: R² 高就代表模型好
**✓ 正确理解**: R² 高只说明自变量与因变量相关性强，但不能证明因果关系。高 R² 也可能是过拟合（尤其在变量多、样本少时）。应结合调整 R² 和残差分析综合判断。

### ❌ 误区 2: 忘记对分类变量创建虚拟变量
**✓ 正确做法**: 分类变量（如性别、地区）不能直接作为数值变量输入回归模型。必须创建虚拟变量（0/1 编码），且 k 个类别只需 k-1 个虚拟变量（避免虚拟变量陷阱）。

### ❌ 误区 3: 忽略多重共线性
**✓ 正确做法**: 在解释回归系数之前，应检查 VIF。如果 VIF > 10，说明自变量之间存在严重共线性，系数估计不可靠。此时应删除高度相关的变量之一，或使用正则化方法。

### ❌ 误区 4: 相关性等于因果性
**✓ 正确理解**: 多元回归只能揭示变量之间的关联关系，不能证明因果关系。要建立因果关系，需要随机对照实验或使用工具变量等因果推断方法。

### ❌ 误区 5: 忽略残差分析
**✓ 正确做法**: 回归模型的有效性依赖于残差的假设（正态性、同方差性、独立性）。如果残差不满足这些假设，系数的标准误和 p 值可能不可靠。应通过残差图、QQ 图等进行诊断。